Load data

In [ ]:
import pandas as pd
import numpy as np
import os
import tiktoken
# Paths are relative to the repo root - run this notebook from there.


In [ ]:
# Load in data
data = pd.read_csv('data_unbalanced_72K.csv')
data

In [ ]:
data['CI_TiAb'] = data['contraindication'] + ". " + data['intervention'] + ". " + data['TiAb']
data

In [ ]:
print(data['CI_TiAb'][0])

In [ ]:
# Get tokens
def count_tokens(text: str, encoding_name="gpt-4") -> int:
    encoding = tiktoken.encoding_for_model(encoding_name)
    num_tokens = len(encoding.encode(text))
    return num_tokens

data['tokens'] = data['CI_TiAb'].apply(lambda x: count_tokens(x))
data


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,12))
sns.histplot(data['tokens'], binwidth=10)
plt.axvline(512, color='r', linestyle='--')  # Add vertical line at 512
plt.text(512, plt.ylim()[1]*0.9, '512 tokens', color='r')  # Add label for the line
plt.xlim(0, 1000)  # Set x-max to 1000
plt.show()



Use group-based split, so validation and final val are on query results unrelated to the train set. 

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Create a GroupShuffleSplit object to split into training and temporary test set 
gss_train_test = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=1)

# Using 'rapport_number' as the groups
groups = data['rapport_number']

# Perform the training and temporary test set split
train_idx, temp_test_idx = next(gss_train_test.split(data, groups=groups))

# Create training and temporary test DataFrames
train_df = data.iloc[train_idx]
temp_test_df = data.iloc[temp_test_idx]

# Now, split the temporary test set into validation and test sets using GroupShuffleSplit with train_size=0.5
gss_val_test = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val_test.split(temp_test_df, groups=temp_test_df['rapport_number']))

# Create validation and test DataFrames
val_df = temp_test_df.iloc[val_idx]
test_df = temp_test_df.iloc[test_idx]


In [ ]:
print("Size of train_df: ", train_df.shape[0])
print("Number of case value equals 1 in train_df: ", train_df[train_df['case'] == 1].shape[0])

print("Size of val_df: ", val_df.shape[0])
print("Number of case value equals 1 in val_df: ", val_df[val_df['case'] == 1].shape[0])

print("Size of test_df: ", test_df.shape[0])
print("Number of case value equals 1 in test_df: ", test_df[test_df['case'] == 1].shape[0])


In [ ]:
train_percentage = train_df.shape[0] / data.shape[0] * 100
val_percentage = val_df.shape[0] / data.shape[0] * 100
test_percentage = test_df.shape[0] / data.shape[0] * 100

print("Percentage of data in train set: ", train_percentage)
print("Percentage of data in validation set: ", val_percentage)
print("Percentage of data in test set: ", test_percentage)

# Create balanced dataframes for model training/validation/testing 
#### Choose random controls matched by number of missing abstracts

In [ ]:
def get_balanced_set(complete_df: pd.DataFrame) -> pd.DataFrame:
    """Function to return a balanced set of cases and controls matched by percentage of title-only"""

    # Step 1: count number of cases without abstract
    n_no_ab = pd.isna(complete_df[complete_df['case'] == 1]['abstract']).sum()
    n_with_ab = complete_df[complete_df['case'] == 1].shape[0] - n_no_ab

    # Step 2: split in cases and control (with and without) df
    df_cases = complete_df[complete_df['case'] == 1]
    df_controls_noAb = complete_df[(complete_df['case'] == 0) & (complete_df['abstract'].isna())]
    df_controls_TiAb = complete_df[(complete_df['case'] == 0) & (complete_df['abstract'].notna())]

    # Step 3: randomly draw a matching set of controls for each case (with and without abstract)
    controls_sample_noAb = df_controls_noAb.sample(n=n_no_ab, random_state=101)
    controls_sample_TiAb = df_controls_TiAb.sample(n=n_with_ab, random_state=101)

    # Step 4: concatenate the two DataFrames to get the balanced dataset
    balanced_df = pd.concat([df_cases, controls_sample_noAb, controls_sample_TiAb], axis=0).reset_index(drop=True)

    # Step 5: shuffle the balanced DataFrame
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    return balanced_df

    

In [ ]:
balanced_df_train = get_balanced_set(train_df)
balanced_df_val = get_balanced_set(val_df)
balanced_df_test = get_balanced_set(test_df)

In [ ]:
for set in [balanced_df_train, balanced_df_val, balanced_df_test]:
    print("Number of articles: ", set.shape[0])
    print("Number of cases: ", sum(set['case']==1))


In [ ]:
# Check if any 'rapport_number' values have leaked between train, val and test set
train_rapport_numbers = balanced_df_train['rapport_number'].unique()
val_rapport_numbers = balanced_df_val['rapport_number'].unique()
test_rapport_numbers = balanced_df_test['rapport_number'].unique()

leakage_train_val = np.intersect1d(train_rapport_numbers, val_rapport_numbers)
leakage_train_test = np.intersect1d(train_rapport_numbers, test_rapport_numbers)
leakage_val_test = np.intersect1d(val_rapport_numbers, test_rapport_numbers)

print("Leakage between train and validation set: ", leakage_train_val)
print("Leakage between train and test set: ", leakage_train_test)
print("Leakage between validation and test set: ", leakage_val_test)


Save balanced and unbalanced dataframes for later use

In [ ]:
# directory for outputs
out = "scripts/pubmed_output"
os.makedirs(out, exist_ok=True)

balanced_df_train.to_csv(f'{out}/healthbase_articles_balanced_train_CI.csv', index=False)
balanced_df_val.to_csv(f'{out}/healthbase_articles_balanced_val_CI.csv', index=False)
balanced_df_test.to_csv(f'{out}/healthbase_articles_balanced_test_CI.csv', index=False)


In [ ]:
# directory for outputs
out = "scripts/pubmed_output"
os.makedirs(out, exist_ok=True)

train_df.to_csv(f'{out}/healthbase_articles_unbalanced_train_CI_55K.csv', index=False)
val_df.to_csv(f'{out}/healthbase_articles_unbalanced_val_CI_11K.csv', index=False)
test_df.to_csv(f'{out}/healthbase_articles_unbalanced_test_CI_7K.csv', index=False)
